In [1]:
# Standard library imports
import os
import sys

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bayes_opt import BayesianOptimization
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Local application imports
sys.path.append(os.path.dirname(os.getcwd()))  # Adjust path to include the top-level directory
from topicminer.utils.email_text_processing import read_json_to_dataframe
from topicminer.utils.statistical_transforms import train_test_split_utility, upsample_classes
from topicminer.config.config import PROCESSED_TEXT_COL, CATEGORIES_COL

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

In [2]:
class AutoLabelEncoder:
    def __init__(self):
        self.encoder = LabelEncoder()
        self.is_encoded = False

    def fit_transform(self, y):
        """Automatically transforms non-numeric labels to numeric."""
        if not self.is_numeric(y):
            y = self.encoder.fit_transform(y)
            self.is_encoded = True
        return y

    def transform(self, y):
        """Transforms labels using the fitted encoder if they were previously encoded."""
        if self.is_encoded:
            return self.encoder.transform(y)
        return y

    def inverse_transform(self, y):
        """Converts numeric labels back to original if they were encoded."""
        if self.is_encoded:
            return self.encoder.inverse_transform(y)
        return y

    @staticmethod
    def is_numeric(y):
        """Check if the label array is numeric."""
        return np.issubdtype(y.dtype, np.number)


In [3]:
def rf_cv(n_estimators, data, targets):
    """
    RandomForest cross-validation.
    
    Performs cross-validation to evaluate a RandomForest model using
    a range of 'n_estimators'. Utilizes Bayesian Optimization to optimize
    the number of estimators for best cross-validation score.
    
    Parameters:
        n_estimators: number of trees in the forest.
        data: features in the dataset.
        targets: target outcomes in the dataset.
    
    Returns:
        Mean of cross-validation scores.
    """
    # Ensure the parameter is an integer because Bayesian Optimization
    # typically works with continuous variables.
    estimator = RandomForestClassifier(n_estimators=int(n_estimators), random_state=42)
    cval = cross_val_score(estimator, data, targets, scoring='accuracy', cv=4)
    return cval.mean()

def optimize_rf(data, targets):
    """
    Apply Bayesian Optimization to RandomForest parameters.
    
    Parameters:
        data: Training feature data.
        targets: Training target data.
    
    Returns:
        Dictionary with information about the best parameters and target score.
    """
    # Define range of values for 'n_estimators'
    param_bounds = {'n_estimators': (10, 500)}  # You can adjust these bounds based on domain knowledge
    
    optimizer = BayesianOptimization(
        f=lambda n_estimators: rf_cv(n_estimators, data, targets),
        pbounds=param_bounds,
        random_state=1,
        verbose=2
    )
    
    optimizer.maximize(init_points=2, n_iter=10)
    
    return optimizer.max

def train_rf_classifier(X_train_bal, y_train_bal):
    """
    Trains a RandomForest classifier using parameters designed to reduce overfitting.
    """
    # Optimize the RF model to find the best parameters (assuming optimization handles this)
    best_params = optimize_rf(X_train_bal, y_train_bal)
    optimal_n_estimators = int(best_params['params']['n_estimators'])  # Round to nearest integer

    # Initialize the RandomForestClassifier with parameters adjusted to combat overfitting
    rf_clf = RandomForestClassifier(
        n_estimators=optimal_n_estimators,
        max_depth=10,                  # Limit the depth of each tree
        min_samples_split=10,          # Minimum number of samples required to split an internal node
        min_samples_leaf=4,            # Minimum number of samples required to be at a leaf node
        random_state=42
    )

    # Fit the model on the balanced training data
    rf_clf.fit(X_train_bal, y_train_bal)

    return rf_clf

def score_rf_classifier(X_train_bal, X_test, y_train_bal, y_test, rf_clf):
    """
    Scores the RandomForest classifier and prints the accuracy and classification report.
    Automatically handles label encoding if necessary.
    
    Parameters:
        X_train_bal, X_test: The training and testing feature data.
        y_train_bal, y_test: The training and testing label data.
        rf_clf: The trained Random Forest classifier.
    """
    converter = AutoLabelEncoder()
    y_train_bal_encoded = converter.fit_transform(y_train_bal)
    y_test_encoded = converter.transform(y_test)

    # Predict and evaluate on the training set
    y_train_pred = rf_clf.predict(X_train_bal)
    train_accuracy = accuracy_score(y_train_bal_encoded, y_train_pred)
    print(f"Accuracy on Training Set: {train_accuracy:.4f}")

    # Predict and evaluate on the test set
    y_test_pred = rf_clf.predict(X_test)
    test_accuracy = accuracy_score(y_test_encoded, y_test_pred)
    print(f"Accuracy on Test Set: {test_accuracy:.4f}")

    # Detailed classification report for the test set
    print("Classification Report for the Test Set:")
    print(classification_report(y_test_encoded, y_test_pred))

    # Optionally, print the original labels if they were encoded
    if converter.is_encoded:
        print("Classification Report with Original Labels:")
        y_test_original = converter.inverse_transform(y_test)
        y_test_pred_original = converter.inverse_transform(y_test_pred)
        print(classification_report(y_test_original, y_test_pred_original))

In [4]:

def evaluate_upsampling_methods(X_train, X_test, y_train, y_test, methods):
    results = {}
    for method in methods:
        print(f"Evaluating {method}...")
        X_train_bal, y_train_bal = upsample_classes(X_train, y_train, method=method)
        rf_clf = train_rf_classifier(X_train_bal, y_train_bal)
        y_pred = rf_clf.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')  # Weighted F1 score for imbalanced classes
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        
        results[method] = {
            'Accuracy': acc,
            'F1 Score': f1,
            'TP': tp,
            'TN': tn,
            'FP': fp,
            'FN': fn,
            'Detailed Report': classification_report(y_test, y_pred)
        }
        
        print(f"Metrics with {method}: Accuracy={acc:.4f}, F1 Score={f1:.4f}, TP={tp}, TN={tn}, FP={fp}, FN={fn}")

    # Find the best method based on a preferred metric, here chosen as F1 score
    best_method = max(results, key=lambda x: results[x]['F1 Score'])
    print(f"Best upsampling method: {best_method} with F1 Score: {results[best_method]['F1 Score']:.4f}")
    return best_method, results

# Example usage
# upsampling_methods = ['duplicate', 'smote', 'adasyn', None]  # Include None to test without any upsampling
# best_method, all_results = evaluate_upsampling_methods(X_train, X_test, y_train, y_test, upsampling_methods)

NameError: name 'X_train' is not defined

In [ ]:
X_train_bal, X_test, y_train_bal, y_test = train_test_split_utility(test_size=0.33, upsampling_method='adasyn')
rf_clf = train_rf_classifier(X_train_bal, y_train_bal)
score_rf_classifier(X_train_bal, X_test, y_train_bal, y_test, rf_clf)



In [ ]:
STOP  

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Assuming the 'best_params' dictionary has been returned from the 'optimize_rf' function
optimal_n_estimators = int(best_params['params']['n_estimators'])  # Round to nearest integer

# Initialize the RandomForestClassifier with the optimal number of trees
clf = RandomForestClassifier(n_estimators=optimal_n_estimators, random_state=42)

# Fit the model on the balanced training data
clf.fit(X_train_bal, y_train_bal)

# Optionally, you can evaluate the model on your test set
from sklearn.metrics import accuracy_score, classification_report

# Predict on the test set
y_pred = clf.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on Test Set: {accuracy:.4f}")

# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialize and fit the logistic regression model
logreg_model = LogisticRegression(max_iter=1000, random_state=42)
logreg_model.fit(X_train_bal, y_train_bal)

# Predict on the training set
y_train_pred = logreg_model.predict(X_train_bal)

# Predict on the test set
y_test_pred = logreg_model.predict(X_test)

# Calculate accuracy on the training set
accuracy_train = accuracy_score(y_train_bal, y_train_pred)
print(f"Accuracy on Training Set: {accuracy_train:.4f}")

# Calculate accuracy on the test set
accuracy_test = accuracy_score(y_test, y_test_pred)
print(f"Accuracy on Test Set: {accuracy_test:.4f}")

# Detailed classification report for the test set
print("Classification Report for the Test Set:")
print(classification_report(y_test, y_test_pred))

# Detailed classification report for the training set
print("Classification Report for the Training Set:")
print(classification_report(y_train_bal, y_train_pred))

In [ ]:

# Here is the code for making the random forrest classifier. IS there a way to mofify the code so it find the optimal number of trees to use?



clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_bal, y_train_bal)




In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Set up a simpler RandomForest model with possible regularization
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf.fit(X_train_bal, y_train_bal)

# Perform cross-validation to get a better sense of true training performance
cv_scores = cross_val_score(clf, X_train_bal, y_train_bal, cv=5, scoring='accuracy')
print(f"Mean CV Accuracy: {np.mean(cv_scores):.4f}")

# Predict and evaluate on the training set
y_train_pred = clf.predict(X_train_bal)
train_accuracy = accuracy_score(y_train_bal, y_train_pred)
print(f"Accuracy on Training Set: {train_accuracy:.4f}")

# Predict and evaluate on the test set
y_test_pred = clf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Accuracy on Test Set: {test_accuracy:.4f}")

# Detailed classification report for the test set
print("Classification Report for the Test Set:")
print(classification_report(y_test, y_test_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score

# Prepare stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# List to store each fold accuracy
accuracies = []

# Perform stratified cross-validation
for train_index, test_index in skf.split(X_train_bal, y_train_bal):
    X_train_fold, X_test_fold = X_train_bal[train_index], X_train_bal[test_index]
    y_train_fold, y_test_fold = y_train_bal[train_index], y_train_bal[test_index]

    # Fit model on each fold
    log_reg.fit(X_train_fold, y_train_fold)
    
    # Evaluate on the test fold
    y_pred_fold = log_reg.predict(X_test_fold)
    fold_accuracy = accuracy_score(y_test_fold, y_pred_fold)
    accuracies.append(fold_accuracy)

# Average cross-validation accuracy
print(f"Mean CV Accuracy: {np.mean(accuracies):.4f}")

# Evaluate on the original test set
y_test_pred = log_reg.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Accuracy on Test Set: {test_accuracy:.4f}")

# Classification report for the test set
print("Classification Report for the Test Set:")
print(classification_report(y_test, y_test_pred))

In [ ]:
test_size = 0.33
emails_df = read_json_to_dataframe(columns=[PROCESSED_TEXT_COL, CATEGORIES_COL])
tfidf = TfidfVectorizer(stop_words='english', max_features=1000)
features = tfidf.fit_transform(emails_df[PROCESSED_TEXT_COL]).toarray()
labels = emails_df[CATEGORIES_COL]
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=test_size, random_state=42)
X_train_bal, y_train_bal = upsample_classes(X_train, y_train)

In [ ]:
print(f"{len(y_train)} - {len(y_train_bal)}" )
print(f"{len(X_train)} - {len(X_train_bal)} ")

In [ ]:
X_train_bal, X_test, y_train_bal, y_test = data_split_utility(test_size=0.33)

In [ ]:

features, labels = preprocess_data(emails_df)

In [ ]:
features